In [3]:
import os
import requests
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import geodatasets
import geopandas as gpd

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from shapely.geometry import Point

In [4]:
MAPILLARY_TOKEN = "MLY|26664131136518032|9af8051ce63bc8d3adb2abc9b889ed15"

DATA_DIR = "data"
IMAGE_DIR = f"{DATA_DIR}/images"
META_PATH = f"{DATA_DIR}/metadata.csv"

NUM_BBOXES = 1500
IMAGES_PER_BBOX = 8
K_REGIONS = 500

BATCH_SIZE = 64
EPOCHS = 1000

ALPHA = 1.0
BETA = 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(IMAGE_DIR, exist_ok=True)

In [5]:
def random_bbox():

    land_regions = [
        (30, 50, -10, 30),
        (25, 48, -125, -70),
        (-35, -15, 115, 150),
        (10, 45, 60, 140)
    ]

    lat_min, lat_max, lon_min, lon_max = land_regions[np.random.randint(len(land_regions))]

    lat = np.random.uniform(lat_min, lat_max)
    lon = np.random.uniform(lon_min, lon_max)

    d = 0.01
    return [lon-d, lat-d, lon+d, lat+d]

In [6]:
class CountryLookup:

    def __init__(self):

        try:
            path = geodatasets.get_path("naturalearth_lowres")
            self.world = gpd.read_file(path)
        except:
            url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
            self.world = gpd.read_file(url)

        if 'name' not in self.world.columns and 'ADMIN' in self.world.columns:
            self.world = self.world.rename(columns={'ADMIN':'name'})

        self.world = self.world[["name","geometry"]]

    def lookup(self, lat, lon):

        point = Point(lon, lat)
        match = self.world[self.world.contains(point)]

        return match.iloc[0]["name"] if not match.empty else "Unknown"

In [7]:
class MapillaryDownloader:

    def __init__(self, token, country_lookup):

        self.token = token
        self.base_url = "https://graph.mapillary.com/images"
        self.headers = {"Authorization": f"OAuth {token}"}
        self.country_lookup = country_lookup

    def fetch_metadata(self, bbox):

        params = {
            "fields": "id,thumb_1024_url,geometry",
            "bbox": ",".join(map(str,bbox)),
            "limit": IMAGES_PER_BBOX
        }

        try:
            r = requests.get(self.base_url, params=params, headers=self.headers, timeout=10)
            return r.json().get("data",[]) if r.status_code == 200 else []
        except:
            return []

    def download_dataset(self):

        records = []

        while len(records) < 1000:

            bbox = random_bbox()
            metadata = self.fetch_metadata(bbox)

            for img in metadata:

                url = img.get("thumb_1024_url")
                if not url:
                    continue

                try:

                    r = requests.get(url, timeout=10)

                    if r.status_code != 200:
                        continue

                    path = f"{IMAGE_DIR}/{img['id']}.jpg"

                    with open(path,"wb") as f:
                        f.write(r.content)

                    lon, lat = img["geometry"]["coordinates"]

                    records.append({
                        "path": path,
                        "lat": lat,
                        "lon": lon,
                        "country": self.country_lookup.lookup(lat,lon)
                    })

                except:
                    continue

        return pd.DataFrame(records)

In [9]:
if os.path.exists(META_PATH):

    print("Loading cached dataset...")
    df = pd.read_csv(META_PATH)

else:

    print("Downloading dataset...")

    cl = CountryLookup()
    dl = MapillaryDownloader(MAPILLARY_TOKEN, cl)

    df = dl.download_dataset()

    df.to_csv(META_PATH, index=False)

print("Dataset size:", len(df))

Loading cached dataset...
Dataset size: 102


In [10]:
def create_regions(df):

    coords = df[["lat","lon"]].values
    k = min(K_REGIONS, len(coords))

    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)

    df["region"] = kmeans.fit_predict(coords)

    return df, kmeans.cluster_centers_


def spatial_split(df):

    df["tile"] = df.lat.astype(int).astype(str) + "_" + df.lon.astype(int).astype(str)

    tiles = df["tile"].unique()

    tr_t, te_t = train_test_split(tiles, test_size=0.2, random_state=42)
    tr_t, va_t = train_test_split(tr_t, test_size=0.2, random_state=42)

    return (
        df[df.tile.isin(tr_t)],
        df[df.tile.isin(va_t)],
        df[df.tile.isin(te_t)]
    )


df, centroids = create_regions(df)

train_df, val_df, test_df = spatial_split(df)

c:\Users\Tyrese\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (99) found smaller than n_clusters (102). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


In [11]:
class GeoDataset(Dataset):

    def __init__(self, df, country_map, transform=None):

        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.country_map = country_map

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
            row = self.df.iloc[idx]
            img = Image.open(row.path).convert("RGB")

            if self.transform:
                img = self.transform(img)

            # Force long (Int64) for CrossEntropy compatibility
            c = torch.tensor(self.country_map.get(row.country, 0)).long()
            r = torch.tensor(row.region).long()

            return img, c, r, torch.tensor(row.lat), torch.tensor(row.lon)

In [12]:
class GeoModel(nn.Module):

    def __init__(self, num_countries, num_regions):

        super().__init__()

        res = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

        self.backbone = nn.Sequential(*list(res.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.c_head = nn.Linear(2048, num_countries)
        self.r_head = nn.Linear(2048, num_regions)

    def forward(self,x):

        x = self.pool(self.backbone(x))
        x = torch.flatten(x,1)

        return self.c_head(x), self.r_head(x)

In [13]:
def haversine(lat1, lon1, lat2, lon2):

    R = 6371.0

    lat1 = torch.deg2rad(lat1)
    lon1 = torch.deg2rad(lon1)
    lat2 = torch.deg2rad(lat2)
    lon2 = torch.deg2rad(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = torch.sin(dlat/2)**2 + torch.cos(lat1)*torch.cos(lat2)*torch.sin(dlon/2)**2
    c = 2 * torch.atan2(torch.sqrt(a), torch.sqrt(1-a))

    return R*c

In [14]:
c_map = {c:i for i,c in enumerate(sorted(df.country.unique()))}

tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

tr_ld = DataLoader(GeoDataset(train_df,c_map,tf),batch_size=BATCH_SIZE,shuffle=True)
va_ld = DataLoader(GeoDataset(val_df,c_map,tf),batch_size=BATCH_SIZE)
te_ld = DataLoader(GeoDataset(test_df,c_map,tf),batch_size=BATCH_SIZE)

In [15]:
model = GeoModel(len(c_map), len(centroids)).to(device)

opt = optim.Adam(model.parameters(), lr=1e-4)
crit = nn.CrossEntropyLoss()

centroids_t = torch.tensor(centroids, dtype=torch.float32).to(device)

In [ ]:
for ep in range(EPOCHS):
    model.train()
    total_loss = 0

    for imgs, c_t, _, lat_t, lon_t in tr_ld:
        imgs = imgs.to(device)
        c_t = c_t.to(device)
        lat_t = lat_t.to(device).float()
        lon_t = lon_t.to(device).float()

        opt.zero_grad()
        
        c_logits, coords = model(imgs)
        pred_lat = coords[:, 0]
        pred_lon = coords[:, 1]

        loss_c = crit(c_logits, c_t)
        loss_h = haversine(pred_lat, pred_lon, lat_t, lon_t).mean()
        
        loss = ALPHA * loss_c + BETA * (loss_h / 6371.0) 

        loss.backward()
        opt.step()
        total_loss += loss.item()

    train_loss = total_loss / len(tr_ld)

    model.eval()
    val_error = 0
    count = 0

    with torch.no_grad():
        for imgs, _, _, lat_t, lon_t in va_ld:
            imgs = imgs.to(device)
            lat_t = lat_t.to(device).float()
            lon_t = lon_t.to(device).float()

            _, coords = model(imgs)
            pred_lat = coords[:, 0]
            pred_lon = coords[:, 1]

            dist = haversine(pred_lat, pred_lon, lat_t, lon_t)
            val_error += dist.sum().item()
            count += len(dist)

    mean_km = val_error / count
    print(f"Epoch {ep} | Train Loss {train_loss:.4f} | Val Haversine {mean_km:.2f} km")

Epoch 0 | Train Loss 3.4593 | Val Haversine 4786.92 km
Epoch 1 | Train Loss 2.5624 | Val Haversine 4780.40 km
Epoch 2 | Train Loss 1.9899 | Val Haversine 4773.24 km
Epoch 3 | Train Loss 1.6010 | Val Haversine 4765.07 km
Epoch 4 | Train Loss 1.3225 | Val Haversine 4756.46 km
Epoch 5 | Train Loss 1.1292 | Val Haversine 4747.46 km
Epoch 6 | Train Loss 1.0027 | Val Haversine 4738.68 km
Epoch 7 | Train Loss 0.9244 | Val Haversine 4730.34 km
Epoch 8 | Train Loss 0.8767 | Val Haversine 4722.67 km
Epoch 9 | Train Loss 0.8476 | Val Haversine 4715.22 km
Epoch 10 | Train Loss 0.8296 | Val Haversine 4707.84 km
Epoch 11 | Train Loss 0.8180 | Val Haversine 4700.43 km
Epoch 12 | Train Loss 0.8102 | Val Haversine 4692.84 km
Epoch 13 | Train Loss 0.8046 | Val Haversine 4685.02 km
Epoch 14 | Train Loss 0.8004 | Val Haversine 4676.92 km
Epoch 15 | Train Loss 0.7970 | Val Haversine 4668.70 km
Epoch 16 | Train Loss 0.7942 | Val Haversine 4660.39 km
Epoch 17 | Train Loss 0.7918 | Val Haversine 4652.00 km
Ep

In [1]:
print('hello')

hello
